# Dataset 1 — Cleaning Notebook

**Dataset:** Compact friction intervention testing  
**Purpose:** Apply all cleaning steps identified during the audit. Produce cleaned output files.  
**Prerequisite:** Run `01_dataset_1_inspection.ipynb` first.  
**Audit report:** `outputs/reports/ds1_audit_report.md`  

**Cleaning steps applied in this notebook:**
1. Load all three CSVs with correct delimiter (`sep=';'`)
2. Drop 1 NaN-participantUuid row
3. Investigate and drop fully duplicate rows (8,302)
4. Convert timestamp to UTC datetime
5. Fix `whastapp` typo → `whatsapp` in new `app_name` column
6. Create `event_outcome_simple` (openedApp→opened, dismissedAppOpening→dismissed, closedApp→closed, NaN→missing)
7. Survey: exclude non-consenting participants
8. Survey: deduplicate (keep first row per participant)
9. Survey: extract numeric prefix from Likert-scale strings
10. Survey: split multi-select fields into binary indicator columns
11. Create participant linkage table
12. Validate outputs
13. Save to `outputs/cleaned/`

**Raw files are never modified.**

## 0. Setup

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

ROOT     = Path("..").resolve()
DS1_RAW  = ROOT / "data_raw" / "1. data set, compact friction intervention testing"
CLEANED  = ROOT / "outputs" / "cleaned"
INTERIM  = ROOT / "outputs" / "interim"
REPORTS  = ROOT / "outputs" / "reports"
CLEANED.mkdir(parents=True, exist_ok=True)
INTERIM.mkdir(parents=True, exist_ok=True)

BEH_CSV  = DS1_RAW / "Behavior data set" / "behavioral_data_set_1.csv"
PRE_CSV  = DS1_RAW / "Survey data sets" / "Pre-survey"  / "pre_survey_data_set_1.csv"
POST_CSV = DS1_RAW / "Survey data sets" / "Post-survey" / "post_survey_data_set_1.csv"

print("Root:", ROOT)
print("Cleaned output dir:", CLEANED)

---
## 1. Load Raw Files

All three CSVs are semicolon-delimited. Always use `sep=';'`.

In [ ]:
beh_raw  = pd.read_csv(BEH_CSV,  sep=";")
pre_raw  = pd.read_csv(PRE_CSV,  sep=";")
post_raw = pd.read_csv(POST_CSV, sep=";")

print(f"Behavioral raw:   {beh_raw.shape}")
print(f"Pre-survey raw:   {pre_raw.shape}")
print(f"Post-survey raw:  {post_raw.shape}")

---
## 2. Behavioral Cleaning
### 2.1 Drop NaN participantUuid row

In [ ]:
mask_null_id = beh_raw["participantUuid"].isna()
print(f"Rows with NaN participantUuid: {mask_null_id.sum()}")
print(beh_raw[mask_null_id].to_string())
print()
beh = beh_raw[~mask_null_id].copy()
print(f"After drop: {len(beh)} rows  (removed {mask_null_id.sum()})")

### 2.2 Deduplicate fully duplicate rows

8,302 rows are exact duplicates across all 6 columns.  
Rule: drop all but the first occurrence (`keep='first'`).  
Rationale: no reliable way to distinguish double-logging from genuine same-second events.

In [ ]:
n_before = len(beh)
n_dupes  = beh.duplicated().sum()
print(f"Duplicate rows before dedup: {n_dupes:,}")

beh = beh.drop_duplicates(keep="first").reset_index(drop=True)
n_after = len(beh)

print(f"Rows removed:  {n_before - n_after:,}")
print(f"Rows remaining: {n_after:,}")

### 2.3 Convert timestamp to UTC datetime

In [ ]:
beh["timestamp_utc"] = pd.to_datetime(beh["timestamp"], unit="s", utc=True)
print("Sample timestamps:")
print(beh[["timestamp", "timestamp_utc"]].head(5).to_string())
print()
print(f"Date range: {beh['timestamp_utc'].min()}  →  {beh['timestamp_utc'].max()}")

### 2.4 Fix app name typo

Create a canonical `app_name` column that corrects `whastapp` → `whatsapp`.  
The original `app` column is preserved.

In [ ]:
typo_map = {"whastapp": "whatsapp"}

beh["app_name"] = beh["app"].replace(typo_map)

# Verify fix
print("Original 'app' column — whatsapp variants:")
print(beh["app"].value_counts()[beh["app"].value_counts().index.str.lower().str.contains("what")])
print()
print("Canonical 'app_name' column — whatsapp variants:")
print(beh["app_name"].value_counts()[beh["app_name"].value_counts().index.str.lower().str.contains("what")])

### 2.5 Create event_outcome_simple

Map the raw `resolution` values to a simplified canonical column for cross-dataset harmonization.

In [ ]:
outcome_map = {
    "openedApp":           "opened",
    "dismissedAppOpening": "dismissed",
    "closedApp":           "closed",
}

beh["event_outcome_simple"] = beh["resolution"].map(outcome_map).fillna("missing")

print("event_outcome_simple value counts:")
print(beh["event_outcome_simple"].value_counts(dropna=False))

### 2.6 Create participant_id_canonical

In [ ]:
beh["participant_id_canonical"] = beh["participantUuid"]
print("Canonical ID column added (copy of participantUuid).")
print(f"Unique participants: {beh['participant_id_canonical'].nunique()}")

### 2.7 Behavioral cleaning summary

In [ ]:
print("=" * 55)
print("BEHAVIORAL CLEANING SUMMARY")
print("=" * 55)
print(f"  Raw rows:                   {len(beh_raw):,}")
print(f"  After drop NaN UUID:        {len(beh_raw[beh_raw['participantUuid'].notna()]):,}")
print(f"  After deduplication:        {len(beh):,}")
print(f"  Rows removed total:         {len(beh_raw) - len(beh):,}")
print(f"  Unique participants:         {beh['participant_id_canonical'].nunique()}")
print(f"  Columns added:              timestamp_utc, app_name, event_outcome_simple, participant_id_canonical")
print()
print("Final columns:")
print(beh.columns.tolist())
print()
print("NaN check on cleaned behavioral data:")
print(beh.isnull().sum())

---
## 3. Pre-Survey Cleaning
### 3.1 Exclude non-consenting participants

In [ ]:
print("consent_general before exclusion:")
print(pre_raw["consent_general"].value_counts(dropna=False))
print()

# F031: consent is a PARTICIPANT-level property, not a row-level one. It used to be
# filtered inside each survey file independently and never applied to the behavioural
# file, so a participant who refused in one file survived in the others (verified:
# 4 non-consenting participants / 1,877 events in ds1_behavioral_clean.csv, 3 rows in
# the pre-survey, 2 in the post-survey). Build one blocklist from BOTH raw surveys and
# apply it to every cleaned output.
def _canon_uuid(s: pd.Series) -> pd.Series:
    """Normalise a participantUuid column for set membership tests."""
    return s.astype(str).str.strip().str.upper()


NON_CONSENT_IDS = set(
    _canon_uuid(pre_raw.loc[pre_raw["consent_general"] == "I do NOT consent", "participantUuid"])
) | set(
    _canon_uuid(post_raw.loc[post_raw["consent_general"] == "I do NOT consent", "participantUuid"])
)
print(f"Participant-level non-consent blocklist: {len(NON_CONSENT_IDS)} participants")

pre = pre_raw[~_canon_uuid(pre_raw["participantUuid"]).isin(NON_CONSENT_IDS)].copy()
print(f"Rows after excluding non-consenting: {len(pre)}  (removed {len(pre_raw)-len(pre)})")


### 3.2 Deduplicate — keep first row per participant

26 participants have multiple pre-survey rows. Rule: keep first occurrence.  
Rationale: first submission is most likely the primary response; re-submissions are edge cases.

In [ ]:
n_before = len(pre)
pre = pre.sort_values("participantUuid").drop_duplicates(subset="participantUuid", keep="first").reset_index(drop=True)
print(f"Rows before dedup: {n_before}")
print(f"Rows after dedup:  {len(pre)}  (removed {n_before - len(pre)} duplicate rows)")
print(f"Unique participants: {pre['participantUuid'].nunique()}")

### 3.3 Extract numeric prefix from Likert-scale columns

Values like `'5 - tendentially problematic'` → integer `5`.  
New columns get suffix `_num`. Original columns are preserved.

In [ ]:
def extract_likert_num(series):
    """Extract leading integer from composite Likert strings like '5 - label'."""
    return series.str.extract(r'^(\d+)', expand=False).astype(float)

likert_pre = ["problematic_cons", "happy", "problematic_cons_general", "problematic_cons_compared"]

for col in likert_pre:
    pre[f"{col}_num"] = extract_likert_num(pre[col])
    print(f"{col}_num:  range {pre[f'{col}_num'].min():.0f}–{pre[f'{col}_num'].max():.0f},  "
          f"NaN={pre[f'{col}_num'].isna().sum()}")

### 3.4 Split multi-select fields into binary indicators

`problems` and `goals` contain comma-separated option labels.  
Each unique option becomes a binary column (1 = selected, 0 = not selected, NaN = question missing).

In [ ]:
def split_multiselect(df, col, prefix):
    """
    Split a comma-concatenated multi-select column into binary indicator columns.
    Returns the original df with new columns appended.
    """
    # Get all unique options (excluding NaN rows)
    all_options = set()
    for val in df[col].dropna():
        for opt in str(val).split(","):
            all_options.add(opt.strip())

    # Create a clean column name from the option label
    def to_colname(opt):
        return prefix + "_" + re.sub(r'[^a-z0-9]+', '_', opt.lower()).strip('_')

    result = df.copy()
    for opt in sorted(all_options):
        cname = to_colname(opt)
        result[cname] = df[col].apply(
            lambda x: 1 if pd.notna(x) and opt in str(x).split(",") else (np.nan if pd.isna(x) else 0)
        )
    return result, sorted(all_options)

pre, problem_options = split_multiselect(pre, "problems", "problem")
pre, goal_options    = split_multiselect(pre, "goals",    "goal")

print("Problem options found:", problem_options)
print()
print("Goal options found:",    goal_options)
print()
new_problem_cols = [c for c in pre.columns if c.startswith("problem_")]
new_goal_cols    = [c for c in pre.columns if c.startswith("goal_")]
print("New binary columns — problems:", new_problem_cols)
print("New binary columns — goals:   ", new_goal_cols)

### 3.5 Parse `hours` as numeric

In [ ]:
# hours is stored as open-format text (e.g. '4', '2-3', '3-4h') — coerce to float
pre["hours_num_raw"] = pd.to_numeric(pre["hours"], errors="coerce")

# F125: the coercion was previously unbounded, so impossible self-reports (max 123 h/day)
# survived into ds1_pre_survey_clean.csv and inflated the column (std 5.83 vs a median of
# 3.0). Hours of use per day cannot exceed 24 by construction, and 0 is not a usable
# self-report of daily use either.
IMPLAUSIBLE = (pre["hours_num_raw"] <= 0) | (pre["hours_num_raw"] > 24)
print(f"Implausible hours values set to NaN ({int(IMPLAUSIBLE.sum())} rows): "
      f"{sorted(pre.loc[IMPLAUSIBLE, 'hours_num_raw'].dropna().tolist())}")
pre["hours_num"] = pre["hours_num_raw"].where(~IMPLAUSIBLE)

print(f"hours_num:  {pre['hours_num'].notna().sum()} non-null,  {pre['hours_num'].isna().sum()} NaN")
print(f"Range:  {pre['hours_num'].min():.2f}-{pre['hours_num'].max():.2f}  h/day")
_n_answered = int((pre["hours"].astype(str).str.strip().replace("nan", "") != "").sum())
print(f"(raw non-blank `hours` answers: {_n_answered}; "
      f"lost to numeric coercion: {_n_answered - int(pre['hours_num_raw'].notna().sum())})")
print()
print("Values that couldn't be parsed as numeric (non-numeric hours entries):")
non_numeric = pre[pre["hours_num_raw"].isna() & pre["hours"].notna()]["hours"]
print(non_numeric.value_counts().head(10))

### 3.6 Add canonical ID

In [ ]:
pre["participant_id_canonical"] = pre["participantUuid"]
print(f"Pre-survey cleaned: {len(pre)} rows, {pre['participant_id_canonical'].nunique()} unique participants")

---
## 4. Post-Survey Cleaning
### 4.1 Exclude non-consenting

In [ ]:
print("consent_general (post-survey) before exclusion:")
print(post_raw["consent_general"].value_counts(dropna=False))
print()
# F031: apply the participant-level blocklist built in section 3.1, not a row-level
# filter on this file alone.
post = post_raw[~_canon_uuid(post_raw["participantUuid"]).isin(NON_CONSENT_IDS)].copy()
print(f"Rows after exclusion: {len(post)}  (removed {len(post_raw)-len(post)})")

### 4.2 Deduplicate — keep first row per participant

In [ ]:
n_before = len(post)
post = post.sort_values("participantUuid").drop_duplicates(subset="participantUuid", keep="first").reset_index(drop=True)
print(f"Rows before dedup: {n_before}")
print(f"Rows after dedup:  {len(post)}  (removed {n_before - len(post)})")

### 4.3 Extract numeric prefix from Likert-scale columns

In [ ]:
likert_post = ["problems_solved", "goals_reached", "problematic_cons",
               "happy", "progress_satisfaction",
               "problematic_cons_general", "problematic_cons_compared"]

for col in likert_post:
    post[f"{col}_num"] = extract_likert_num(post[col])
    print(f"{col}_num:  range {post[f'{col}_num'].min():.0f}–{post[f'{col}_num'].max():.0f},  "
          f"NaN={post[f'{col}_num'].isna().sum()}")

### 4.4 Parse `hours` as numeric

In [ ]:
post["hours_num"] = pd.to_numeric(post["hours"], errors="coerce")
print(f"hours_num: {post['hours_num'].notna().sum()} non-null, {post['hours_num'].isna().sum()} NaN")

### 4.5 Add canonical ID

In [ ]:
post["participant_id_canonical"] = post["participantUuid"]
print(f"Post-survey cleaned: {len(post)} rows, {post['participant_id_canonical'].nunique()} unique participants")

### 4.6 Apply the participant-level consent blocklist to the behavioural frame

**F031.** Consent was previously filtered per survey file only, so 4 non-consenting
participants (1,877 events) remained in `ds1_behavioral_clean.csv`. The blocklist built in
section 3.1 is applied here, before linkage and saving, so all four cleaned outputs agree.


In [ ]:
# F031: consent applies to the participant, so remove blocklisted participants from the
# behavioural frame too (previously never consent-filtered at all).
n_beh_before = len(beh)
n_parts_before = beh["participant_id_canonical"].nunique()
beh = beh[~_canon_uuid(beh["participantUuid"]).isin(NON_CONSENT_IDS)].copy()
print(f"Behavioral rows removed by consent blocklist: {n_beh_before - len(beh):,}")
print(f"Participants: {n_parts_before} -> {beh['participant_id_canonical'].nunique()}")

assert not _canon_uuid(beh["participantUuid"]).isin(NON_CONSENT_IDS).any(), \
    "FAIL: non-consenting participants remain in cleaned behavioral"
assert not _canon_uuid(pre["participantUuid"]).isin(NON_CONSENT_IDS).any(), \
    "FAIL: non-consenting participants remain in cleaned pre-survey"
assert not _canon_uuid(post["participantUuid"]).isin(NON_CONSENT_IDS).any(), \
    "FAIL: non-consenting participants remain in cleaned post-survey"


---
## 5. Participant Linkage Table

Build a table of all unique participant IDs across all three files,  
with flags indicating which files they appear in.

In [ ]:
beh_ids  = set(beh["participant_id_canonical"])
pre_ids  = set(pre["participant_id_canonical"])
post_ids = set(post["participant_id_canonical"])
all_ids  = beh_ids | pre_ids | post_ids

linkage = pd.DataFrame({
    "participant_id_canonical": list(all_ids),
})
linkage["in_behavioral"] = linkage["participant_id_canonical"].isin(beh_ids).astype(int)
linkage["in_pre_survey"] = linkage["participant_id_canonical"].isin(pre_ids).astype(int)
linkage["in_post_survey"]= linkage["participant_id_canonical"].isin(post_ids).astype(int)
linkage["n_files"]       = linkage[["in_behavioral","in_pre_survey","in_post_survey"]].sum(axis=1)
linkage = linkage.sort_values("participant_id_canonical").reset_index(drop=True)

print("Linkage table shape:", linkage.shape)
print()
print("Participants by file coverage:")
print(linkage["n_files"].value_counts().sort_index())
print()
print("All-three subset:", (linkage["n_files"] == 3).sum())
print()
linkage.head()

---
## 6. Validation Checks

Verify the cleaned outputs meet expectations before saving.

In [ ]:
print("=" * 55)
print("VALIDATION — BEHAVIORAL")
print("=" * 55)
assert beh["participantUuid"].notna().all(), "FAIL: NaN participantUuid in cleaned behavioral"
assert beh.duplicated().sum() == 0, "FAIL: duplicate rows remain in cleaned behavioral"
assert beh["timestamp_utc"].notna().all(), "FAIL: NaN timestamp_utc in cleaned behavioral"
assert set(beh["event_outcome_simple"].unique()) <= {"opened","dismissed","closed","missing"}, "FAIL: unexpected outcome values"
assert "whastapp" not in beh["app_name"].values, "FAIL: typo 'whastapp' still present in app_name"
assert "whatsapp" in beh["app_name"].values, "FAIL: 'whatsapp' not found in app_name"
print("All behavioral checks passed.")
print(f"  Rows: {len(beh):,}")
print(f"  Unique participants: {beh['participant_id_canonical'].nunique()}")
print(f"  event_outcome_simple: {beh['event_outcome_simple'].value_counts().to_dict()}")

In [ ]:
print("=" * 55)
print("VALIDATION — PRE-SURVEY")
print("=" * 55)
assert pre.duplicated(subset="participantUuid").sum() == 0, "FAIL: duplicate participantUuid in pre-survey"
assert (pre["consent_general"] != "I do NOT consent").all(), "FAIL: non-consenting rows remain in pre-survey"
for col in ["problematic_cons_num", "happy_num", "problematic_cons_general_num", "problematic_cons_compared_num"]:
    valid = pre[col].dropna()
    assert valid.between(1, 7).all(), f"FAIL: {col} has out-of-range values"
# F125: hours_num is bounded to (0, 24] h/day by the plausibility gate in section 3.5.
assert pre["hours_num"].dropna().between(0, 24, inclusive="right").all(), "FAIL: hours_num out of (0, 24]"
print("All pre-survey checks passed.")
print(f"  Rows: {len(pre)}, Unique participants: {pre['participantUuid'].nunique()}")

In [ ]:
print("=" * 55)
print("VALIDATION — POST-SURVEY")
print("=" * 55)
assert post.duplicated(subset="participantUuid").sum() == 0, "FAIL: duplicate participantUuid in post-survey"
assert (post["consent_general"] != "I do NOT consent").all(), "FAIL: non-consenting rows remain in post-survey"
print("All post-survey checks passed.")
print(f"  Rows: {len(post)}, Unique participants: {post['participantUuid'].nunique()}")

---
## 7. Save Cleaned Outputs

Save to `outputs/cleaned/` as CSV files.  
Raw files in `data_raw/` are never modified.

In [ ]:
BEH_OUT    = CLEANED / "ds1_behavioral_clean.csv"
PRE_OUT    = CLEANED / "ds1_pre_survey_clean.csv"
POST_OUT   = CLEANED / "ds1_post_survey_clean.csv"
LINK_OUT   = CLEANED / "ds1_participant_linkage.csv"

beh.to_csv(BEH_OUT,  index=False)
pre.to_csv(PRE_OUT,  index=False)
post.to_csv(POST_OUT, index=False)
linkage.to_csv(LINK_OUT, index=False)

print("Saved:")
for p in [BEH_OUT, PRE_OUT, POST_OUT, LINK_OUT]:
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name:<45}  {size_kb:,.0f} KB")

---
## 8. Final Cleaning Summary

In [ ]:
print("=" * 65)
print("DS1 CLEANING COMPLETE")
print("=" * 65)
print()
print("BEHAVIORAL")
print(f"  Raw rows:                {len(beh_raw):,}")
print(f"  Cleaned rows:            {len(beh):,}")
print(f"  Removed:                 {len(beh_raw)-len(beh):,}  (1 NaN-UUID + duplicate rows)")
print(f"  Unique participants:     {beh['participant_id_canonical'].nunique()}")
print(f"  New columns:             timestamp_utc, app_name, event_outcome_simple, participant_id_canonical")
print()
print("PRE-SURVEY")
print(f"  Raw rows:                {len(pre_raw):,}")
print(f"  Cleaned rows:            {len(pre):,}")
print(f"  Removed:                 {len(pre_raw)-len(pre):,}  (non-consent + duplicates)")
print(f"  Unique participants:     {pre['participant_id_canonical'].nunique()}")
print(f"  New columns:             *_num (Likert), problem_* / goal_* (binary), hours_num, participant_id_canonical")
print()
print("POST-SURVEY")
print(f"  Raw rows:                {len(post_raw):,}")
print(f"  Cleaned rows:            {len(post):,}")
print(f"  Removed:                 {len(post_raw)-len(post):,}  (non-consent + duplicates)")
print(f"  Unique participants:     {post['participant_id_canonical'].nunique()}")
print(f"  New columns:             *_num (Likert), hours_num, participant_id_canonical")
print()
print("PARTICIPANT COVERAGE")
print(f"  Total unique IDs:        {len(linkage):,}")
print(f"  In all three files:      {(linkage['n_files']==3).sum()}")
print(f"  In behavioral only:      {((linkage['in_behavioral']==1) & (linkage['n_files']==1)).sum()}")
print(f"  In pre only:             {((linkage['in_pre_survey']==1) & (linkage['n_files']==1)).sum()}")
print()
print("OUTPUT FILES")
for p in [BEH_OUT, PRE_OUT, POST_OUT, LINK_OUT]:
    print(f"  outputs/cleaned/{p.name}")